In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Dataset path
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DataCoSupplyChainDataset.csv"
)

# Load the dataset
df = pd.read_csv(
    DATA_PATH,
    encoding="latin1" # As the DataCo CSV may contain text characters that cause UTF-8 decoding errors. latin1 is a practical way to load this file.
)

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully!
Rows: 180,519
Columns: 53


In [3]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [5]:
df.tail()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
180514,CASH,4,4,40.000000,399.980011,Shipping on time,0,45,Fishing,Brooklyn,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 3:40,Standard Class
180515,DEBIT,3,2,-613.770019,395.980011,Late delivery,1,45,Fishing,Bakersfield,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/19/2016 1:34,Second Class
180516,TRANSFER,5,4,141.110001,391.980011,Late delivery,1,45,Fishing,Bristol,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 21:00,Standard Class
180517,PAYMENT,3,4,186.229996,387.980011,Advance shipping,0,45,Fishing,Caguas,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/18/2016 20:18,Standard Class
180518,PAYMENT,4,4,168.949997,383.980011,Shipping on time,0,45,Fishing,Caguas,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/19/2016 18:54,Standard Class


In [7]:
df.shape

(180519, 53)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

In [10]:
missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

Product Description    180519
Order Zipcode          155679
Customer Lname              8
Customer Zipcode            3
dtype: int64

In [12]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [13]:
TARGET = "Late_delivery_risk"

print(df[TARGET].value_counts(dropna=False))

print("\nProportion:")
print(
    df[TARGET]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Proportion:
Late_delivery_risk
1    54.83
0    45.17
Name: proportion, dtype: float64


In [15]:
leakage_candidates = [
    "Delivery Status",
    "Days for shipping (real)",
    "Late_delivery_risk",
    "shipping date (DateOrders)",
    "Order Status",
    "order date (DateOrders)",
    "Days for shipment (scheduled)"
]

existing = [
    col for col in leakage_candidates
    if col in df.columns
]

df[existing].head(10)

,Delivery Status,Days for shipping (real),Late_delivery_risk,shipping date (DateOrders),Order Status,order date (DateOrders),Days for shipment (scheduled)
0,Advance shipping,3,0,2/3/2018 22:56,COMPLETE,1/31/2018 22:56,4
1,Late delivery,5,1,1/18/2018 12:27,PENDING,1/13/2018 12:27,4
2,Shipping on time,4,0,1/17/2018 12:06,CLOSED,1/13/2018 12:06,4
3,Advance shipping,3,0,1/16/2018 11:45,COMPLETE,1/13/2018 11:45,4
4,Advance shipping,2,0,1/15/2018 11:24,PENDING_PAYMENT,1/13/2018 11:24,4
5,Shipping canceled,6,0,1/19/2018 11:03,CANCELED,1/13/2018 11:03,4
6,Late delivery,2,1,1/15/2018 10:42,COMPLETE,1/13/2018 10:42,1
7,Late delivery,2,1,1/15/2018 10:21,PROCESSING,1/13/2018 10:21,1
8,Late delivery,3,1,1/16/2018 10:00,CLOSED,1/13/2018 10:00,2
9,Late delivery,2,1,1/15/2018 9:39,CLOSED,1/13/2018 9:39,1


In [16]:
quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (
        df.isna().mean() * 100
    ).round(2),
    "unique_values": df.nunique(dropna=False)
})

quality_report = quality_report.sort_values(
    by="missing_percent",
    ascending=False
)

quality_report

,data_type,missing_count,missing_percent,unique_values
Product Description,float64,180519,100.00,1
Order Zipcode,float64,155679,86.24,610
Type,str,0,0.00,4
Order Profit Per Order,float64,0,0.00,21998
Order Id,int64,0,0.00,65752
Order Item Cardprod Id,int64,0,0.00,118
Order Item Discount,float64,0,0.00,1017
Order Item Discount Rate,float64,0,0.00,18
Order Item Id,int64,0,0.00,180519
Order Item Product Price,float64,0,0.00,75


In [17]:
REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "data_quality_report.csv"
)

quality_report.to_csv(REPORT_PATH)

print(f"Report saved to: {REPORT_PATH}")

Report saved to: /Users/Kushagra/Desktop/Code/LogiCast/reports/data_quality_report.csv


In [23]:
df[TARGET].value_counts(dropna=False)

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64